# Descriptive Analysis of products.csv
Exploratory overview of the grocery products dataset: structure, cleanliness, summary statistics, category distributions, and pricing/discount insights.

## 1. Import Libraries and Configure Environment
Using pandas/numpy for data handling and seaborn/matplotlib for quick visuals.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Load products.csv into DataFrame
Load the CSV and preview shape for a quick sanity check.

In [ ]:
DATA_PATH = "products.csv"

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()

print(f"Loaded shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())

## 3. Inspect Schema, Head, and Data Types
Peek at the head, random sample, and column metadata to understand the schema.

In [ ]:
print("Column names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

print("\nHead:")
display(df.head())

print("\nRandom sample:")
display(df.sample(min(5, len(df)), random_state=RANDOM_SEED))

print("\nUnique counts per column:")
display(df.nunique().to_frame("unique_values"))

## 4. Assess and Handle Missing Values
Check for gaps and visualize completeness.

In [ ]:
na_counts = df.isna().sum()
na_pct = (na_counts / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": na_counts, "missing_pct": na_pct})

print("Missing summary:")
display(missing_df)

plt.figure(figsize=(8, 4))
sns.heatmap(df.isna(), cbar=False)
plt.title("Missing Data Heatmap")
plt.show()

## 5. Compute Descriptive Statistics (Numeric Features)
Convert pricing/discount fields to numeric where needed and compute summary metrics.

In [ ]:
if df["discount"].dtype == object:
    df["discount_pct"] = df["discount"].str.rstrip("% ").astype(float)
else:
    df["discount_pct"] = df["discount"].astype(float)

for col in ["original_price", "discounted_price"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["savings"] = df["original_price"] - df["discounted_price"]
df["discount_ratio"] = df["discounted_price"] / df["original_price"]

df["is_essential"] = (
    df["is_essential"].astype(str).str.lower().isin(["true", "1", "yes"])
)

numeric_cols = ["original_price", "discounted_price", "discount_pct", "savings", "discount_ratio"]

print("Numeric summary:")
display(df[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75]).T)

print("\nSkewness:")
display(df[numeric_cols].skew().to_frame("skew"))

## 6. Analyze Categorical Distributions
Identify dominant categories and essential vs non-essential split.

In [ ]:
cat_counts = df["category"].value_counts().to_frame("count")
cat_counts["pct"] = (cat_counts["count"] / len(df) * 100).round(2)

print("Category distribution:")
display(cat_counts)

essential_counts = df["is_essential"].value_counts().rename_axis("is_essential").to_frame("count")
essential_counts["pct"] = (essential_counts["count"] / len(df) * 100).round(2)

print("\nEssential vs non-essential:")
display(essential_counts)

## 7. Visualize Numeric Distributions
Inspect spread/outliers for key numeric fields.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_cols = ["original_price", "discounted_price", "discount_pct", "savings"]

for ax, col in zip(axes.flat, plot_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax, color="#4c72b0")
    ax.set_title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(data=df[plot_cols], orient="h", palette="Set2")
plt.title("Boxplots of Price/Discount Features")
plt.show()

## 8. Correlation Analysis for Numeric Features
Evaluate relationships between pricing and discount measures.

In [ ]:
numeric_cols = ["original_price", "discounted_price", "discount_pct", "savings", "discount_ratio"]

corr = df[numeric_cols].corr()
print("Correlation matrix:")
display(corr)

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap (Numeric Features)")
plt.show()

## 9. Save Cleaned Data or Summary Outputs
Persist cleaned numeric fields for downstream tasks.

In [ ]:
# Uncomment to persist cleaned dataset
# df.to_csv("products_cleaned.csv", index=False)

# Example: export category summary
# (cat_counts.reset_index().rename(columns={"index": "category"})
#  .to_csv("category_summary.csv", index=False))

print("Ready for downstream use; uncomment saves above to export.")